# 1. 패키지 설치

In [ ]:
# uv add pypdf=">=4.2.0,<5.0.0"
# uv add langchain-upstage

# 2. 환경변수 불러오기

- `.env` 파일에 `UPSTAGE_API_KEY` 등록

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[:4])

up_9


# 3. LLM 답변 생성

- Upstage Console에서 발급받은 API Key를 `UPSTAGE_API_KEY`라고 저장하면 별도의 설정 없이 `ChatUpstage`를 사용할 수 있음

In [2]:
from langchain_upstage import ChatUpstage

#llm = ChatUpstage(temperature=0.5)
llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm.model_name)

c:\myclass\ai_langchain\mylangchin-uv-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


solar-pro3


In [3]:
ai_message=llm.invoke("LangChain은 무엇인가요?")
print(type(ai_message))
print(ai_message.content)

<class 'langchain_core.messages.ai.AIMessage'>
**LangChain**은 대규모 언어 모델(LLM)을 활용해 **응용 프로그램**을 쉽게 만들 수 있게 해 주는 **프레임워크**입니다.  
“언어 체인(Language Chain)”이라는 이름 그대로, 여러 LLM 호출·데이터 소스·외부 도구를 **연결(체인)** 하여 복잡한 작업을 단계별로 수행하도록 설계돼 있습니다.

---

## 핵심 개념

| 개념 | 설명 |
|------|------|
| **Chain** | 두 개 이상의 LLM 호출, 프롬프트, 혹은 외부 연산을 순차적으로 연결하는 기본 단위. 예: `PromptChain`, `LLMChain`, `SequentialChain` 등 |
| **Prompt** | LLM에 전달하는 텍스트 템플릿. 변수 치환, few‑shot 예시, 역할 지정 등을 포함해 원하는 출력 형식을 정의 |
| **Document** | 텍스트, PDF, HTML, CSV 등 **데이터 소스**를 의미. `DocumentLoaders` 로 추출하고, `VectorStore` 로 저장해 검색에 사용 |
| **Memory** | 대화형 애플리케이션에서 이전 대화·상태를 기억하게 하는 메커니즘. `ConversationBufferMemory`, `ConversationSummaryMemory` 등 |
| **Agents** | LLM이 **도구(함수, API, DB 등)** 를 스스로 선택하고 호출하도록 하는 시스템. 예: `ChatOpenAI` + `Tool` + `AgentExecutor` |
| **Retrievers** | 저장된 문서 집합에서 **relevant** 문서를 찾아주는 검색 엔진. `VectorStoreRetriever`, `ElasticsearchRetriever` 등 |
| **Output Parsers** | LLM이 반환한 텍스트를 **구조화**(JSON, CSV, XML 등)하거나 정규화해서 downstrea

In [4]:
# using chat stream
for chunk in llm.stream("LangChain은 무엇인가요?"):
    print(chunk.content, end="")

**LangChain**은 **대규모 언어 모델(LLM)**을 활용해 **복잡한 자연어 처리(NLP) 애플리케이션**을 빠르게 구축하고 연결하기 위한 **오픈소스 프레임워크**입니다.  

---

## 핵심 개념

| 요소 | 설명 |
|------|------|
| **Chain** | 여러 단계(입력 → 전처리 → LLM 호출 → 후처리)를 순차적으로 연결한 파이프라인. 예: `PromptTemplate → LLM → OutputParser`. |
| **Agent** | LLM이 스스로 **행동을 선택**하고 **도구(Tool)를 호출**해 문제를 해결하도록 하는 시스템. 예: 검색, 데이터베이스 조회, 외부 API 호출 등. |
| **Memory** | 대화형 애플리케이션에서 **이전 대화 기록**을 저장·검색해 컨텍스트를 유지하도록 돕는 기능. |
| **Prompt Templates** | 재사용 가능한 프롬프트 템플릿을 정의해, 동적 변수와 포맷을 쉽게 삽입. |
| **Document Loaders & Text Splitters** | PDF, DOCX, HTML 등 다양한 형식의 문서를 **로드**하고 **청크(chunk)로 분할**해 LLM에 효율적으로 전달. |
| **Vector Stores** | 텍스트 청크를 **임베딩**해 벡터 DB에 저장하고, 유사도 검색을 통해 Retrieval‑Augmented Generation(RAG) 구현. |
| **Callbacks & Tracing** | 실행 흐름을 로깅·시각화하고, 디버깅·성능 모니터링을 지원. |
| **Integration** | OpenAI, Anthropic, Google Gemini, Hugging Face, Azure OpenAI 등 다양한 LLM 제공자와 **플러그‑인** 형태로 연결. |

---

## 왜 LangChain을 쓰나요?

1. **재사용성** – 프롬프트, 체인, 에이전트 등을 모듈화해 여러 프로젝트에서 그대로 사용 가능.  
2. **유연성** 

In [ ]:
from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate

translation_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a professional translator specializing in Korean-English translation."),
        ("human", "Translate this from {source_lang} to {target_lang}: {text}")
    ])

llm = ChatUpstage(
        model="solar-pro3",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )

# 체인 실행
chain = translation_prompt | llm

response = chain.invoke({
    "source_lang": "English",
    "target_lang": "Korean", 
    "text": "LangChain is a powerful framework for building AI applications."
})

print("Upstage Response:")
print(response.content)
    

Upstage Response:
"LangChain은 AI 애플리케이션 구축을 위한 강력한 프레임워크입니다."  

### 추가 설명 (필요시 참고):  
- "Powerful framework"은 맥락에 따라 "강력한 프레임워크" 또는 "고성능 프레임워크"로 번역 가능합니다.  
- "Building AI applications"은 "AI 애플리케이션 개발"로도 자연스럽게 표현할 수 있으나, 원문의 동사 형태("building")를 살려 "구축"으로 번역했습니다.  
- 기술 용어인 "LangChain"은 음차 표기 없이 그대로 사용하는 것이 일반적입니다.  

예시 변형:  
- "LangChain은 AI 애플리케이션 개발을 위한 고성능 프레임워크입니다."  
- "LangChain은 AI 애플리케이션 구축에 최적화된 강력한 도구입니다." (의미 확장 시)


In [6]:
from langchain_core.prompts import ChatPromptTemplate

llm = ChatUpstage(
        model="solar-pro2",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )

# using chain
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that translates English to Korean."),
        ("human", "Translate this sentence from English to Korean. {english_text}."),
    ]
)

llm = ChatUpstage()
chain = prompt | llm

ai_message=chain.invoke({"english_text": "Hello, How are you?"})
print(ai_message.content)

이 문장을 한국어로 번역해줘. Hello, How are you?.


#### Groundness Check
* Groundedness Check API는 사용자가 제공한 Context(컨텍스트)에 대한 AI 어시스턴트의 응답이 실제로 그 컨텍스트에 기반하고 있는지 여부를 확인합니다.
* 최신버전에서는 UpstageGroundednessCheck 는 deprecated 되었음

In [8]:
# from langchain_upstage import UpstageGroundednessCheck

# groundedness_check = UpstageGroundednessCheck()

# request_input = {
#     "context": "삼성전자는 연결 기준으로 매출 74.07조원, 영업이익 10.44조원의 2024년 2분기 실적을 발표했다. 전사 매출은 전분기 대비 3% 증가한 74.07조원을 기록했다. DS부문은 메모리 업황 회복으로 전분기 대비 23% 증가하고, SDC는 OLED 판매 호조로 증가했다.",
#     "answer": "삼성전자의 2024년 2분기 매출은 약 74.07조원이다.",
# }

# response = groundedness_check.invoke(request_input)
# print(response)  


#### UpstageEmbeddings

In [7]:
from langchain_upstage import UpstageEmbeddings
import json

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")
doc_result = embeddings.embed_documents(
    ["Sung is a professor.", "This is another document"]
)
print("📄 문서 임베딩 (첫 번째):")
print(f"   타입: {type(doc_result[0])}")  # <class 'list'>
print(f"   길이: {len(doc_result[0])}")   # 차원 수 (예: 4096)
print(f"   처음 10개: {doc_result[0][:10]}...\n")  # 처음 10개만 출력

query_result = embeddings.embed_query("What does Sung do?")
print(" 쿼리 임베딩:")
print(f"   타입: {type(query_result)}")
print(f"   길이: {len(query_result)}")

# 🔧 문자열로 변환 4가지 방법
print("\n 문자열 변환 옵션:")

# 1. JSON 문자열 (가장 읽기 쉬움)
json_str = json.dumps(query_result[:10] + ["..."])  # 처음 10개만
print(f"   1. JSON: {json_str}")

# 2. 간단 리스트 문자열
list_str = str(query_result[:10]) + "..."
print(f"   2. 리스트: {list_str}")

# 3. 공백으로 구분 (공백 벡터)
space_str = " ".join(map(str, query_result[:20]))
print(f"   3. 공백: {space_str[:100]}...")

# 4. 16진수 (hex) 문자열 (컴팩트)
hex_str = "".join(f"{int(x):02x}" for x in query_result[:10])
print(f"   4. Hex: {hex_str}")

# 5. 차원 요약 (실용적)
print(f"   5. 요약: {len(query_result)}차원 벡터")
print(f"   처음 5개: [{', '.join(map(str, query_result[:5]))}]")

📄 문서 임베딩 (첫 번째):
   타입: <class 'list'>
   길이: 4096
   처음 10개: [-0.003814697265625, -0.028656005859375, 0.0061187744140625, 0.0158233642578125, -0.01361083984375, -0.01751708984375, 0.006534576416015625, -0.00797271728515625, 0.0011148452758789062, 0.01152801513671875]...

 쿼리 임베딩:
   타입: <class 'list'>
   길이: 4096

 문자열 변환 옵션:
   1. JSON: [0.01364898681640625, -0.04339599609375, 0.0016193389892578125, 0.013031005859375, -0.031494140625, -0.01763916015625, 0.0088958740234375, -0.00789642333984375, -0.00742340087890625, 0.0108642578125, "..."]
   2. 리스트: [0.01364898681640625, -0.04339599609375, 0.0016193389892578125, 0.013031005859375, -0.031494140625, -0.01763916015625, 0.0088958740234375, -0.00789642333984375, -0.00742340087890625, 0.0108642578125]...
   3. 공백: 0.01364898681640625 -0.04339599609375 0.0016193389892578125 0.013031005859375 -0.031494140625 -0.017...
   4. Hex: 00000000000000000000
   5. 요약: 4096차원 벡터
   처음 5개: [0.01364898681640625, -0.04339599609375, 0.0016193389892578125

#### UpstageDocumentParseLoader

In [2]:
from langchain_upstage import UpstageDocumentParseLoader

file_path = "../data/tutorial-korean.pdf"
layzer = UpstageDocumentParseLoader(file_path, split="page")

# For improved memory efficiency, consider using the lazy_load method to load documents page by page.
docs = layzer.load()  # or layzer.lazy_load()
print(len(docs)) #[Document, Document]

for doc in docs[:3]:
    print(type(doc)) #Document
    print(doc.page_content)

39
<class 'langchain_core.documents.base.Document'>
<figure id='0'><img alt="" data-coord="top-left:(558,313); bottom-right:(675,411)" /></figure> <br><p id='1' data-category='paragraph' style='font-size:20px'>BlueJ 튜토리얼</p> <br><p id='2' data-category='paragraph' style='font-size:16px'>한국어 버전 2.0<br>BlueJ Version 2.0.x 用</p> <p id='3' data-category='paragraph' style='font-size:16px'>English Version 2.0.1<br>Michael Kölling<br>Mærsk Insitute<br>University of Southern Denmark</p> <p id='4' data-category='paragraph' style='font-size:16px'>Korean Version 2.0</p> <br><p id='5' data-category='paragraph' style='font-size:16px'>황석형 교수<br>Project Member</p> <br><p id='6' data-category='list' style='font-size:16px'>Ver. 1.0 : 강석진, 민상호, 오경묵, 유형순, 이승환.<br>Ver. 2.0 : 강원준</p> <br><p id='7' data-category='paragraph' style='font-size:16px'>선문대학교 컴퓨터정보학부</p> <figure id='8'><img alt="" data-coord="top-left:(546,1329); bottom-right:(693,1471)" /></figure> <footer id='9' style='font-size:14px'>1</footer>